# Water Mass Division Overview

This notebook visualizes time-series trends of various water quality parameters across the Seto Inland Sea, grouped by **Water Mass Division**.

It loads the primary cleaned dataset (`data/processed/merged_gdf.feather`) and provides interactive widgets to filter the data by Season and Layer before generating the comparative grid plots.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display
import sys
from pathlib import Path

# Ensure local scripts are importable
project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.paths import ProjectPaths

import warnings
warnings.filterwarnings('ignore')

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Data Loading

We load the global `merged_gdf.feather` file. This dataset contains all stations, seasons, layers, and multiple water quality parameters.

In [2]:
# Load the primary dataset
data_path = ProjectPaths.MERGED_DATA_FILE
print(f"Loading data from {data_path}...")
merged_gdf = gpd.read_feather(data_path)
print(f"Data loaded. Shape: {merged_gdf.shape}")

# Ensure year is numeric for plotting
merged_gdf['year'] = pd.to_numeric(merged_gdf['year'], errors='coerce')

# Extract robust lookup of Division mapping
# (Useful to keep colours consistent even if a specific subset misses a division)
station_division = (
    merged_gdf[['zettaicode', 'division']]
    .dropna()
    .drop_duplicates('zettaicode')
    .set_index('zettaicode')['division']
)
all_divisions = sorted(station_division.unique())
palette = sns.color_palette("tab10", n_colors=len(all_divisions))
global_div_colour = dict(zip(all_divisions, palette))

Loading data from /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...
Data loaded. Shape: (42326, 65)


## Interactive Parameter Exploration

Select the **Season**, **Layer** (`saisuipoin`), and **Parameter** below. The plot will update automatically to display the yearly average time series of that parameter, separated into subplots by `division`.

In [3]:
def plot_parameter_by_division(season, layer_val, parameter):
    # 1. Filter Data
    subset = merged_gdf[
        (merged_gdf['season'] == season) & 
        (merged_gdf['saisuipoin'] == layer_val)
    ].copy()
    
    if subset.empty:
        print(f"No data found for Season: {season}, Layer: {layer_val}")
        return
        
    # 2. Aggregate to yearly means per station
    # Note: We group by year and zettaicode, then take the mean of the target parameter.
    yearly_param = (
        subset.groupby(['zettaicode', 'year'])[parameter]
        .mean()
        .reset_index()
    )
    
    # Append the division info
    yearly_param['division'] = yearly_param['zettaicode'].map(station_division)
    
    # Filter out records where the parameter is NaN
    yearly_param = yearly_param.dropna(subset=[parameter, 'division'])
    
    if yearly_param.empty:
        print(f"No numeric data found for parameter '{parameter}' in this subset.")
        return

    # 3. Figure Setup
    # Determine which divisions actually exist in this specific subset
    divisions_present = sorted(yearly_param['division'].unique())
    
    ncols = 2
    nrows = -(-len(divisions_present) // ncols)  # Ceiling division
    
    fig, axes = plt.subplots(nrows, ncols, 
                             figsize=(14, 4 * nrows), 
                             sharey=False, sharex=True)
    
    # Ensure axes is iterable even if there is only 1 subplot
    if nrows * ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
        
    # 4. Plotting
    for ax, div in zip(axes, divisions_present):
        div_data = yearly_param[yearly_param['division'] == div]
        stations_in_div = div_data['zettaicode'].unique()
        
        for station in stations_in_div:
            stn_data = div_data[div_data['zettaicode'] == station]
            # Sort by year to ensure clean lines
            stn_data = stn_data.sort_values('year')
            
            ax.plot(
                stn_data['year'], stn_data[parameter],
                color=global_div_colour.get(div, 'grey'),
                alpha=0.55, linewidth=1.0, label=station
            )
            
        ax.set_title(f'Division: {div} (n={len(stations_in_div)} stations)', 
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Year', fontsize=9)
        ax.set_ylabel(parameter, fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.tick_params(axis='x', rotation=45)
        
    # Hide any unused subplots
    for ax in axes[len(divisions_present):]:
        ax.set_visible(False)

    layer_name = 'Surface' if layer_val == 1 else 'Bottom'
    fig.suptitle(f'{parameter} Time Series by Division ({season}, {layer_name})',
                 fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

# Create Interactive Widgets
season_dropdown = widgets.Dropdown(
    options=['Spring', 'Summer', 'Autumn', 'Winter'],
    value='Summer',
    description='Season:'
)

layer_dropdown = widgets.Dropdown(
    options=[('Surface (1)', 1), ('Bottom (2)', 2)],
    value=2,
    description='Layer:'
)

parameter_dropdown = widgets.Dropdown(
    options=['do_', 'temp', 'salt', 'ph', 'cod', 'tn', 'tp', 'chlorophyl', 'clearness', 'depth'],
    value='do_',
    description='Parameter:'
)

out = widgets.interactive_output(plot_parameter_by_division, {
    'season': season_dropdown,
    'layer_val': layer_dropdown,
    'parameter': parameter_dropdown
})

display(widgets.HBox([season_dropdown, layer_dropdown, parameter_dropdown]))
display(out)

Output()